In [0]:
# ================== Third take home assignment GR5069 Spring 2026 ========

# ================== Global Definitions ====================================

# # Packages to import
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import os 
import matplotlib.pyplot as plt
import seaborn as sns
import tempfile
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier

# Data Paths 
pitstops_path = "/Volumes/gr5069/raw/f1_data/pit_stops.csv"
results_path = "/Volumes/gr5069/raw/f1_data/results.csv"

##### 1. Build any model of your choice with tunable hyperparameters
---------

Model Description: The model I have built is a random forest classification model using ```RandomForestClassifier()```. This model uses the feature variables ```avg_duration, num_stops, first_pit_lap, and grid ``` to determine whether an individual placed in any position that places them on the podium or not. 

The target variable is ```podium```. ```podium = 1``` means that a driver placed in first, second, or third in a race. ```podium = 0``` means that a driver did not place in first, second, or third. I have removed rows where a driver did not finish the race. 


In [0]:
# Load, join and format data

# Load pitstop data 
df_pitstops = pd.read_csv(pitstops_path)
df_pitstops["raceId"] = df_pitstops["raceId"].astype(int)
df_pitstops["driverId"] = df_pitstops["driverId"].astype(int)
df_pitstops["lap"] = df_pitstops["lap"].astype(int)
df_pitstops["stop"] = df_pitstops["stop"].astype(int)
df_pitstops.head()

# Load results data 
df_results = pd.read_csv(results_path)
df_results["raceId"] = df_results["raceId"].astype(int)
df_results["driverId"] = df_results["driverId"].astype(int)

# Join results and pitstop data on raceId and driverId
df_pitstops = df_pitstops.merge(df_results[["raceId", "driverId", "position", "grid"]], on=["raceId", "driverId"], how="left")

# Replace null values (those who did not finish the race) with NaN 
df_pitstops["position"] = df_pitstops["position"].replace("\\N", pd.NA)
df_pitstops["position"] = pd.to_numeric(df_pitstops["position"], errors="coerce")

# Replace null values in grid with NaN
df_pitstops["grid"] = df_pitstops["grid"].replace("\\N", pd.NA)
df_pitstops["grid"] = pd.to_numeric(df_pitstops["grid"], errors="coerce")

In [0]:
# Calculate feature and target variables for model training

# Aggregate features and target into one dataset
df = df_pitstops.groupby(["raceId", "driverId"]).agg(
    avg_duration=("milliseconds", "mean"), # average pitstop duration
    num_stops=("stop", "max"), # number of pitstops
    first_pit_lap=("lap", "min"), # first pitstop lap
    position=("position", "first"), # final position
    grid=("grid", "first") # starting grid position
).reset_index() 

# Create binary target variable — 1 if podium (top 3), 0 otherwise
df["podium"] = (df["position"] <= 3).astype(int)

# Drop rows with missing values (This model will not include drivers who did not finish the race)
df = df.dropna()

# Drop position column because it's no longer needed 
df = df.drop(columns=["position"])

# Display model dataset 
display(df.head(5))

In [0]:
# Separate feature and target variables into trainings and testing data 

# Define feature and target variables (X and y)
y = df["podium"]
X = df.drop(columns=["podium", "raceId", "driverId"])

# Train/test/split (defaults to .75/.25 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 123)

In [0]:

# Basic RF classifier model (with no defined parameters)
with mlflow.start_run(run_name="Baseline Model") as run:

  # Create model, train it, and create predictions
  rf = RandomForestClassifier()
  rf.fit(X_train, y_train)
  predicted_vals = rf.predict(X_test)
  
  # Log model
  mlflow.sklearn.log_model(rf, "random-forest-classifier-model")
  
  # Create metrics
  f1 = f1_score(y_test, predicted_vals)
  accuracy = accuracy_score(y_test, predicted_vals)
  precision = precision_score(y_test, predicted_vals)
  recall = recall_score(y_test, predicted_vals)

  # Log metrics 
  mlflow.log_metric("f1", f1)
  mlflow.log_metric("accuracy", accuracy)
  mlflow.log_metric("precision", precision)
  mlflow.log_metric("recall", recall)
  
  runID = run.info.run_id
  experimentID = run.info.experiment_id
  
  print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))

##### 2. Create an experiment setup where - for each run - you log:
 
- the hyperparameters used in the model
- the model itself
- every possible metric from the model you chose
- at least two artifacts (plots, or csv files)


In [0]:
#Define function that logs parameters, metrics, and feature importance when running your model. 
 
def rf_class(experimentID, run_name, params, X_train, X_test, y_train, y_test):

    with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:

        rfc = RandomForestClassifier(**params)
        rfc.fit(X_train, y_train)
        predicted_vals = rfc.predict(X_test)

        mlflow.sklearn.log_model(rfc, "Basic RF classifier model")
        runID = run.info.run_id
        experimentID = run.info.experiment_id

        [mlflow.log_param(param, value) for param, value in params.items()]

        # Calculate metrics 
        f1 = f1_score(y_test, predicted_vals)
        accuracy = accuracy_score(y_test, predicted_vals)
        precision = precision_score(y_test, predicted_vals)
        recall = recall_score(y_test, predicted_vals)

        # Log metrics 
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)

        # Create feature importance 
        importance = pd.DataFrame(list(zip(X_train.columns, rfc.feature_importances_)),
        columns=["Feature", "Importance"]).sort_values("Importance", ascending=False)

        # Log importances using a temporary file
        temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv", delete=False)
        temp_name = temp.name
        try:
            importance.to_csv(temp_name, index=False)
            mlflow.log_artifact(temp_name, "feature-importance.csv")
        finally:
            temp.close()

        # Create plot
        fig, ax = plt.subplots()

        ConfusionMatrixDisplay.from_predictions(y_test, predicted_vals, ax=ax)
        plt.title("Confusion Matrix — Podium Prediction")

        # Log plot using a temporary file 
        temp = tempfile.NamedTemporaryFile(prefix="confusion-matrix-", suffix=".png")
        temp_name = temp.name
        try:
            fig.savefig(temp_name)
            mlflow.log_artifact(temp_name, "confusion-matrix.png")
        finally:
            temp.close()

        display(fig)
    return run.info.run_id

##### 3. Track your MLFlow experiment and run at least 10 experiments with different parameters each


In [0]:
# 1st experiment with updated hyperparameters
params = {
    "n_estimators": 100,
    "max_depth": 5,
    "random_state": 42,
    "class_weight": "balanced"
}

rf_class(experimentID, "1st Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters for 2nd experiment 

params = {
    "n_estimators": 150,
    "max_depth": 5,
    "random_state": 42,
    "class_weight": "balanced"
}

rf_class(experimentID, "2nd Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters for 3rd experiment 
params = {
    "n_estimators": 100,
    "max_depth": 7,
    "random_state": 42,
    "class_weight": "balanced"
}

rf_class(experimentID, "3rd Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters for 4th experiment  
params = {
    "n_estimators": 100,
    "max_depth": 9,
    "random_state": 42,
    "class_weight": "balanced"
}

rf_class(experimentID, "4th Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters for 5th experiment 
params = {
    "n_estimators": 150,
    "max_depth": 9,
    "random_state": 42,
    "class_weight": "balanced"
}

rf_class(experimentID, "5th Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 6th run 

params = {
    "n_estimators": 100,
    "max_depth": 9,
    "random_state": 42
}

rf_class(experimentID, "6th Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 7th run 

params = {
    "n_estimators": 150,
    "max_depth": 3,
    "random_state": 42
}

rf_class(experimentID, "7th Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 8th run 

params = {
    "n_estimators": 200,
    "max_depth": 7,
    "random_state": 42
}

rf_class(experimentID, "8th Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 9th run 

params = {
    "n_estimators": 50,
    "max_depth": 5,
    "random_state": 42
}

rf_class(experimentID, "9th Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 10th run 

params = {
    "n_estimators": 100,
    "max_depth": 12,
    "random_state": 42,
    "class_weight": "balanced"
}

rf_class(experimentID, "10th Run", params, X_train, X_test, y_train, y_test)

##### 4. Select your best model run and explain why:
---------

I've chosen my 5th run model with the following hyperparameters: 

<br>

``` 

params = {
    "n_estimators": 150,       # Number of trees
    "max_depth": 9,            # Size of each tree
    "random_state": 42, 
    "class_weight": "balanced" # used because the dataset itself is imbalanced
}

```

- Accuracy score: 0.8830
- F1 score: 0.7042  

This model accurately classified 88% of the observations in this dataset. I am choosing to use F1 as the metric to determine which model to choose because my target variable in the dataset is imbalance. For each race, only 3 observations are ``` podium = 1``` .  Imbalanced data can show high accuracy scores even with poorly performing classifiers. F1 score is more reliable for imbalanced data and will be better for determining if my model will be able to accurately classify unseen data. 